# CitizenTech · Vovó Tech — Módulo RPA + IA (Curadoria Automatizada de Saúde Pública)

**Integrantes:**
* Davi Barboza — RM 562370
* Luca Schmidt — RM 560255
* Rickelmy Omine — RM 563286

## 1. Descrição do Problema e Papel do Robô
O assistente de voz **Vovó Tech** orienta pessoas idosas sobre serviços do SUS, vacinação e medicamentos gratuitos (ODS 3 — Saúde e Bem-Estar). No protótipo inicial, a base de conhecimento (`data/kb.md`) possuía apenas 10 trechos manuais e falhava quando o idoso usava termos populares (ex.: *"remédio de pressão"* em vez de *"hipertensão"*). Este robô automatiza de ponta a ponta a coleta em portais públicos do SUS, contorna bloqueios básicos de navegação e usa um Agente de IA para traduzir regras burocráticas em linguagem simples com sinônimos populares.

## 2. Como Executar e Configurar Chaves
1. No menu lateral esquerdo do Google Colab, abra **Secrets (ícone de chave)** e cadastre `GEMINI_API_KEY` com acesso habilitado ao notebook (nenhuma credencial fica salva no código).
2. Clique em **Ambiente de execução → Executar tudo**.
3. O fluxo instala as dependências, gera a fila de fontes (`dados/fontes_sus.csv`), executa a coleta resiliente, grava os logs (`logs/rpa_curadoria.log`) e exporta a base atualizada (`data/kb_expandida.md`).

## 3. Quadro de Temas da Disciplina
| Tema de RPA + IA | Onde atua no fluxo | O que resolve na prática |
| :--- | :--- | :--- |
| **1. Automação Web Avançada e Contorno de Proteções (Playwright + Stealth)** | Etapa `coletar_portal()` | Renderiza páginas dinâmicas (`gov.br`), aplica *User-Agent* e *headers* reais de navegador e reduz bloqueios anti-bot sem violar acesso público. |
| **2. Agente de IA / NLP para Acessibilidade e Sinônimos** | Etapa `agente_curador_ia()` | Converte textos técnicos do SUS em blocos `##` curtos para idosos e injeta sinônimos populares para melhorar a recuperação no RAG. |
| **3. Orquestração em Lote e Validação de Schema (Pandas)** | Etapa `executar_pipeline()` | Valida URLs e parâmetros antes da execução, processa lotes em fila e gera saídas estruturadas (`kb_expandida.md` e `relatorio_rpa.csv`). |
| **4. Resiliência, Exceções e Observabilidade (Logging + Retry/Fallback)** | Transversal (`logs/rpa_curadoria.log`) | Trata erros HTTP (403/404/429), *timeouts* e falhas de IA com *retry* exponencial e caminho seguro sem interromper o lote. |

## 4. Proteção de Dados, Ética e Impacto Social (ODS)
O módulo RPA + IA do **CitizenTech · Vovó Tech** opera sob *Privacy by Design*, coletando exclusivamente informações públicas e institucionais de portais oficiais de saúde (`gov.br`), sem trafegar dados pessoais ou clínicos de cidadãos em APIs externas, em conformidade com a LGPD (Lei nº 13.709/2018). Eticamente, cada trecho gerado preserva a URL de origem e a data de extração para rastreabilidade, contando com *fallback* e auditoria em log para evitar orientações incorretas. Dessa forma, a solução apoia o **ODS 3 (Saúde e Bem-Estar, meta 3.8)** e o **ODS 10**, mantendo atualizadas e acessíveis informações essenciais à longevidade e autonomia da pessoa idosa.

In [1]:
!pip install -q playwright playwright-stealth beautifulsoup4 pandas google-genai
!playwright install chromium
!playwright install-deps chromium

199.3 MiB [] 0% 410.6s199.3 MiB [] 0% 58.5s199.3 MiB [] 0% 15.7s199.3 MiB [] 0% 12.9s199.3 MiB [] 0% 7.8s199.3 MiB [] 1% 5.2s199.3 MiB [] 2% 4.9s199.3 MiB [] 2% 4.6s199.3 MiB [] 3% 4.2s199.3 MiB [] 3% 4.1s199.3 MiB [] 3% 4.2s199.3 MiB [] 4% 4.4s199.3 MiB [] 5% 4.5s199.3 MiB [] 5% 4.6s199.3 MiB [] 6% 4.5s199.3 MiB [] 7% 4.2s199.3 MiB [] 8% 4.1s199.3 MiB [] 8% 4.2s199.3 MiB [] 8% 4.3s199.3 MiB [] 8% 4.5s199.3 MiB [] 9% 4.4s199.3 MiB [] 9% 4.3s199.3 MiB [] 10% 4.1s199.3 MiB [] 11% 4.0s199.3 MiB [] 11% 3.9s199.3 MiB [] 11% 4.0s199.3 MiB [] 12% 4.0s199.3 MiB [] 12% 3.9s199.3 MiB [] 13% 3.9s199.3 MiB [] 13% 3.8s199.3 MiB [] 14% 3.7s199.3 MiB [] 15% 3.7s199.3 MiB [] 15% 3.8s199.3 MiB [] 15% 3.7s199.3 MiB [] 16% 3.7s199.3 MiB [] 17% 3.7s199.3 MiB [] 18% 3.6s199.3 MiB [] 19% 3.5s199.3 MiB [] 19% 3.4s199.3 MiB [] 20% 3.4s199.3 MiB [] 20% 3.3s199.3 MiB [] 21% 3.3s199.3 MiB [] 21% 3.2s199.3 MiB [] 22% 3.2s199.3 MiB [] 23% 3.1s199.3 MiB [] 24% 3.1s199.3 MiB [] 25% 3.1s199.3 MiB [] 26% 3.0s199.3 MiB

In [2]:
import os
import pandas as pd

os.makedirs("dados", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("eval", exist_ok=True)

fontes = [
    {"id": "SUS-01", "tema": "Farmácia Popular e Remédios Gratuitos", "sinonimos": "remédio de pressão, diabetes, asma, farmácia de graça", "url": "https://www.gov.br/saude/pt-br/composicao/sectics/farmacia-popular"},
    {"id": "SUS-02", "tema": "Vacinação do Idoso e Calendário Nacional", "sinonimos": "vacina da gripe, covid, pneumonia, posto de saúde", "url": "https://www.gov.br/saude/pt-br/vacinacao"},
    {"id": "SUS-03", "tema": "Meu SUS Digital e Cartão Nacional de Saúde", "sinonimos": "cartão do sus, cns, histórico de exames, carteira de vacina", "url": "https://www.gov.br/saude/pt-br/composicao/seidigi/meususdigital"},
    {"id": "SUS-04", "tema": "Atenção Primária e Unidade Básica de Saúde (UBS)", "sinonimos": "postinho, consulta de rotina, acompanhamento preventivo, longevidade", "url": "https://www.gov.br/saude/pt-br/composicao/saps"},
    {"id": "SUS-05", "tema": "Urgência e Emergência (UPA 24h e SAMU 192)", "sinonimos": "pronto socorro, ambulância, emergência, dor no peito", "url": "https://www.gov.br/saude/pt-br/composicao/saes/upa"},
    {"id": "SUS-06", "tema": "Saúde da Pessoa Idosa e Envelhecimento Ativo", "sinonimos": "caderneta do idoso, prevenção de quedas, terceira idade, longevidade", "url": "https://www.gov.br/saude/pt-br/assuntos/saude-de-a-a-z/s/saude-da-pessoa-idosa"},
    {"id": "SUS-07", "tema": "Falha Forçada 1 — Página Oficial Inexistente (HTTP 404)", "sinonimos": "teste resiliência página removida", "url": "https://www.gov.br/saude/pt-br/pagina-inexistente-teste-rpa-404"},
    {"id": "SUS-08", "tema": "Falha Forçada 2 — Registro com URL Inválida na Planilha", "sinonimos": "teste validação entrada corrompida", "url": "url_invalida_sem_protocolo"}
]

df_fontes = pd.DataFrame(fontes)
df_fontes.to_csv("dados/fontes_sus.csv", index=False)
print(f"Arquivo dados/fontes_sus.csv gerado com {len(df_fontes)} registros (6 válidos + 2 falhas simuladas).")
df_fontes[["id", "tema", "url"]]

Arquivo dados/fontes_sus.csv gerado com 8 registros (6 válidos + 2 falhas simuladas).


,id,tema,url
0,SUS-01,Farmácia Popular e Remédios Gratuitos,https://www.gov.br/saude/pt-br/composicao/sect...
1,SUS-02,Vacinação do Idoso e Calendário Nacional,https://www.gov.br/saude/pt-br/vacinacao
2,SUS-03,Meu SUS Digital e Cartão Nacional de Saúde,https://www.gov.br/saude/pt-br/composicao/seid...
3,SUS-04,Atenção Primária e Unidade Básica de Saúde (UBS),https://www.gov.br/saude/pt-br/composicao/saps
4,SUS-05,Urgência e Emergência (UPA 24h e SAMU 192),https://www.gov.br/saude/pt-br/composicao/saes...
5,SUS-06,Saúde da Pessoa Idosa e Envelhecimento Ativo,https://www.gov.br/saude/pt-br/assuntos/saude-...
6,SUS-07,Falha Forçada 1 — Página Oficial Inexistente (...,https://www.gov.br/saude/pt-br/pagina-inexiste...
7,SUS-08,Falha Forçada 2 — Registro com URL Inválida na...,url_invalida_sem_protocolo


In [3]:
import asyncio
import logging
import time
from datetime import datetime
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

# Configuração de Logs (arquivo + saída no notebook)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("logs/rpa_curadoria.log", mode="w", encoding="utf-8"),
        logging.StreamHandler()
    ],
    force=True
)

# Recupera chave de forma segura via Colab Secrets (sem hardcoded)
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY")

def url_valida(url: str) -> bool:
    parsed = urlparse(str(url).strip())
    return parsed.scheme in ("http", "https") and bool(parsed.netloc)

def fallback_local(tema: str, sinonimos: str, url: str, motivo: str) -> str:
    """Caminho seguro caso o portal bloqueie, retorne erro ou a IA falhe."""
    data_hoje = datetime.now().strftime("%d/%m/%Y")
    return (
        f"## {tema}\n"
        f"**Palavras-chave populares:** {sinonimos}.\n"
        f"**Orientação oficial (Contingência):** Consulte a Unidade Básica de Saúde (UBS) mais próxima "
        f"levando documento com foto e Cartão SUS, ou ligue gratuitamente para o Disque Saúde 136.\n"
        f"*[Fonte: {url} | Verificado em {data_hoje} | Modo: Fallback ({motivo})]*\n"
    )

def agente_curador_ia(tema: str, sinonimos: str, texto_bruto: str, url: str) -> tuple[str, str]:
    """Usa Agente LLM para sintetizar regras do SUS para idosos; aciona fallback se falhar."""
    data_hoje = datetime.now().strftime("%d/%m/%Y")
    if not API_KEY:
        logging.warning(f"[{tema}] GEMINI_API_KEY não configurada nos Secrets. Acionando fallback seguro.")
        return fallback_local(tema, sinonimos, url, "Sem chave de API"), "fallback_sem_chave"

    try:
        from google import genai
        client = genai.Client(api_key=API_KEY)
        prompt = (
            f"Você é um curador de saúde pública do projeto CitizenTech Vovó Tech.\n"
            f"Com base APENAS no texto oficial extraído abaixo, escreva um bloco curto (máximo 4 frases) "
            f"em linguagem simples para uma pessoa idosa sobre '{tema}'.\n"
            f"Inclua naturalmente os termos populares: {sinonimos}.\n"
            f"Não invente regras e não dê diagnóstico médico.\n\n"
            f"TEXTO OFICIAL:\n{texto_bruto[:3500]}"
        )
        resp = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
        conteudo = (resp.text or "").strip()
        if len(conteudo) < 40:
            raise ValueError("Resposta da IA vazia ou insuficiente.")

        bloco = (
            f"## {tema}\n"
            f"**Palavras-chave populares:** {sinonimos}.\n"
            f"{conteudo}\n"
            f"*[Fonte: {url} | Extraído via RPA+IA em {data_hoje}]*\n"
        )
        return bloco, "sucesso_ia"
    except Exception as e:
        logging.error(f"[{tema}] Falha na API de IA ({type(e).__name__}: {e}). Acionando fallback.")
        return fallback_local(tema, sinonimos, url, f"Erro IA: {type(e).__name__}"), "fallback_erro_ia"

async def coletar_com_stealth(browser, url: str, max_tentativas: int = 2) -> str:
    """Navega em portal protegido com Playwright + Stealth, timeout e retry."""
    context = await browser.new_context(
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
        locale="pt-BR",
        timezone_id="America/Sao_Paulo",
        extra_http_headers={"Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8"}
    )
    page = await context.new_page()
    await page.route("**/*.{png,jpg,jpeg,gif,svg,woff,woff2}", lambda route: route.abort())

    try:
        from playwright_stealth import stealth_async
        await stealth_async(page)
    except Exception:
        pass

    try:
        for tentativa in range(1, max_tentativas + 1):
            try:
                response = await page.goto(url, wait_until="domcontentloaded", timeout=15000)
                status = response.status if response else 0
                if status >= 400:
                    raise RuntimeError(f"HTTP {status}")
                await page.wait_for_timeout(1200)
                html = await page.content()
                soup = BeautifulSoup(html, "html.parser")
                for tag in soup(["script", "style", "nav", "footer", "header"]):
                    tag.decompose()
                texto = " ".join(soup.get_text(separator=" ").split())
                if len(texto) < 150:
                    raise ValueError("Conteúdo extraído muito curto ou bloqueado por anti-bot.")
                return texto
            except Exception as e:
                logging.warning(f"Tentativa {tentativa}/{max_tentativas} falhou para {url}: {e}")
                if tentativa < max_tentativas:
                    await asyncio.sleep(2 * tentativa)
                else:
                    raise
    finally:
        await context.close()

async def executar_pipeline():
    df = pd.read_csv("dados/fontes_sus.csv")
    resultados = []
    blocos_kb = ["# Base de Conhecimento Curada — CitizenTech · Vovó Tech\n\n"]

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled", "--no-sandbox"]
        )
        for _, row in df.iterrows():
            inicio = time.time()
            id_item, tema, sinonimos, url = row["id"], row["tema"], row["sinonimos"], row["url"]
            logging.info(f"Iniciando {id_item} | {tema}")

            if not url_valida(url):
                logging.error(f"{id_item} | Dado inválido detectado na entrada: '{url}'. Tratado com fallback.")
                bloco = fallback_local(tema, sinonimos, url, "URL inválida na entrada")
                status_final = "falha_tratada_validacao"
            else:
                try:
                    texto_pagina = await coletar_com_stealth(browser, url)
                    bloco, status_final = agente_curador_ia(tema, sinonimos, texto_pagina, url)
                    logging.info(f"{id_item} | Concluído com status: {status_final}")
                except Exception as e:
                    logging.error(f"{id_item} | Falha de navegação tratada ({e}). Acionando fallback seguro.")
                    bloco = fallback_local(tema, sinonimos, url, f"Navegação: {e}")
                    status_final = "falha_tratada_web"

            duracao = round(time.time() - inicio, 2)
            blocos_kb.append(bloco + "\n---\n\n")
            resultados.append({
                "id": id_item,
                "tema": tema,
                "status": status_final,
                "tempo_s": duracao
            })

        await browser.close()

    with open("data/kb_expandida.md", "w", encoding="utf-8") as f:
        f.writelines(blocos_kb)

    df_res = pd.DataFrame(resultados)
    df_res.to_csv("eval/relatorio_rpa.csv", index=False)
    return df_res

df_resultado = await executar_pipeline()

2026-11-19 22:36:14,290 | INFO | Iniciando SUS-01 | Farmácia Popular e Remédios Gratuitos
2026-11-19 22:36:23,096 | INFO | AFC is enabled with max remote calls: 10.
2026-11-19 22:36:28,851 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-11-19 22:36:28,854 | INFO | SUS-01 | Concluído com status: sucesso_ia
2026-11-19 22:36:28,856 | INFO | Iniciando SUS-02 | Vacinação do Idoso e Calendário Nacional
2026-11-19 22:36:34,228 | INFO | AFC is enabled with max remote calls: 10.
2026-11-19 22:36:53,686 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-11-19 22:36:53,689 | INFO | SUS-02 | Concluído com status: sucesso_ia
2026-11-19 22:36:53,691 | INFO | Iniciando SUS-03 | Meu SUS Digital e Cartão Nacional de Saúde
2026-11-19 22:36:57,638 | INFO | AFC is enabled with max remote calls: 10.
2026-11-19 22:37:06,665 | INF

In [4]:
print("=== 1. MÉTRICAS DE VALIDAÇÃO DO LOTE ===")
display(df_resultado)
print("\nResumo por status:")
print(df_resultado["status"].value_counts().to_string())
print(f"\nTempo médio por item: {df_resultado['tempo_s'].mean():.2f} s")

print("\n=== 2. EVIDÊNCIA DO LOG (INCLUINDO FALHAS FORÇADAS TRATADAS) ===")
with open("logs/rpa_curadoria.log", "r", encoding="utf-8") as f:
    print(f.read())

print("=== 3. AMOSTRA DA BASE GERADA PARA O VOVÓ TECH (data/kb_expandida.md) ===")
with open("data/kb_expandida.md", "r", encoding="utf-8") as f:
    print(f.read()[:1500])

=== 1. MÉTRICAS DE VALIDAÇÃO DO LOTE ===


,id,tema,status,tempo_s
0,SUS-01,Farmácia Popular e Remédios Gratuitos,sucesso_ia,14.57
1,SUS-02,Vacinação do Idoso e Calendário Nacional,sucesso_ia,24.83
2,SUS-03,Meu SUS Digital e Cartão Nacional de Saúde,sucesso_ia,12.98
3,SUS-04,Atenção Primária e Unidade Básica de Saúde (UBS),sucesso_ia,13.08
4,SUS-05,Urgência e Emergência (UPA 24h e SAMU 192),falha_tratada_web,6.80
5,SUS-06,Saúde da Pessoa Idosa e Envelhecimento Ativo,sucesso_ia,19.56
6,SUS-07,Falha Forçada 1 — Página Oficial Inexistente (...,falha_tratada_web,6.48
7,SUS-08,Falha Forçada 2 — Registro com URL Inválida na...,falha_tratada_validacao,0.01



Resumo por status:
status
sucesso_ia                 5
falha_tratada_web          2
falha_tratada_validacao    1

Tempo médio por item: 12.29 s

=== 2. EVIDÊNCIA DO LOG (INCLUINDO FALHAS FORÇADAS TRATADAS) ===
2026-11-19 22:36:14,290 | INFO | Iniciando SUS-01 | Farmácia Popular e Remédios Gratuitos
2026-11-19 22:36:23,096 | INFO | AFC is enabled with max remote calls: 10.
2026-11-19 22:36:28,851 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-11-19 22:36:28,854 | INFO | SUS-01 | Concluído com status: sucesso_ia
2026-11-19 22:36:28,856 | INFO | Iniciando SUS-02 | Vacinação do Idoso e Calendário Nacional
2026-11-19 22:36:34,228 | INFO | AFC is enabled with max remote calls: 10.
2026-11-19 22:36:53,686 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-11-19 22:36:53,689 | INFO | SUS-02 | Concluído com status: